# Task 10 — Predict endpoint

**Wave 2** · target file: `backend/main.py` · prerequisites: task 04, task 05, task 09

**🎯 Goal:** `POST /predict` — wire image + optional tabular into a prediction.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [ ]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

## What to build
Add `POST /predict` (`response_model=PredictResponse`) to `backend/main.py`. Steps:
1. signature: `file: UploadFile = File(...)`, `ra: float = Form(...)`, `dec: float = Form(...)`, `tabular: str = Form(None)`.
2. read the `.npy` cutout from the upload bytes (`np.load` on a `BytesIO`); unreadable -> `HTTPException(422)`.
3. `preprocess_image(arr)`; catch `ValueError` -> `HTTPException(422)`.
4. if a `tabular` JSON string was sent: `TabularInput(**json.loads(tabular)).model_dump()` -> `tabular_features(row)[0]`, shape it `(1,16)`.
5. `z = predict_z(imgs, feats)[0]`.
6. `return PredictResponse(z=z, distance_gly=z_to_distance_gly(z))`.

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [ ]:
# You add the route to backend/main.py (it needs the app from task 07). The Check cell below
# exercises it. Follow the 6 steps in the spec above; the request is multipart, not JSON.
print("implement POST /predict in backend/main.py, then run Check")

## ➡️ Move it into `backend/main.py`
Once the cell above passes, put your implementation into **`backend/main.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [ ]:
import os, io, json, joblib, numpy as np, tempfile, importlib
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30) * 0.4), "features": list(range(16))}, f"{TMP}/b.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/i.pkl")
os.environ["BASELINE_PATH"] = f"{TMP}/b.pkl"; os.environ["IMAGE_MODEL_PATH"] = f"{TMP}/i.pkl"
import backend.config, backend.model, backend.main
for m in (backend.config, backend.model, backend.main): importlib.reload(m)
from fastapi.testclient import TestClient
c = TestClient(backend.main.app)
def npy(a):
    b = io.BytesIO(); np.save(b, a); b.seek(0); return b
img = np.random.rand(64, 64, 5).astype("float32")
assert c.post("/predict", files={"file": ("c.npy", npy(img))}, data={"ra": 1, "dec": 2}).status_code == 200
tab = json.dumps({"dered_g": 18., "dered_r": 17.5, "petroR50_r": 1.5, "petroR90_r": 3.5})
assert c.post("/predict", files={"file": ("c.npy", npy(img))}, data={"ra": 1, "dec": 2, "tabular": tab}).status_code == 200
assert c.post("/predict", files={"file": ("c.npy", npy(np.zeros((32, 32, 5))))}, data={"ra": 1, "dec": 2}).status_code == 422
print("POST /predict OK")

> ⚠️ **07 + 10 both edit `backend/main.py`.** 07 creates the app + `GET /`; 10 adds `POST /predict`. Keep *both* — see `MERGE.md`.

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/10-predict-endpoint
!git add backend/main.py notebooks/tasks-2026-6-17/10-predict-endpoint/task.ipynb
!git commit -m "10-predict-endpoint: Predict endpoint"
!git push -u origin task/10-predict-endpoint
# then merge back into 2026.6.17 -> see MERGE.md in this folder